# Test Build Swim DB (line by line)

Run each cell to debug the build_swim_db pipeline. First, all data is downloaded to CSVs for inspection.

In [3]:
# 1. Setup paths and load competition_ids.csv
import csv
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
RAW = PROJECT_ROOT / "data" / "raw"
DEBUG_OUT = RAW / "debug_build_swim_db"
DEBUG_OUT.mkdir(parents=True, exist_ok=True)

COMPETITION_IDS_CSV = RAW / "world_aquatics_competition_ids.csv"

# Load competition IDs (same logic as build_swim_db)
pairs = []
with COMPETITION_IDS_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        mid = (row.get("meet_id") or "").strip()
        cid = (row.get("competition_id") or "").strip()
        if mid and cid:
            pairs.append((mid, cid))

# Save competition_ids to debug folder
with (DEBUG_OUT / "competition_ids.csv").open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["meet_id", "competition_id"])
    w.writerows(pairs)

print(f"Loaded {len(pairs)} competitions with IDs")
print(pairs[15])

Loaded 28 competitions with IDs
('wc_2011', '517')


In [4]:
# 2. Fetch events for ONE competition and save to JSON
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from src.scraping.world_aquatics_api import get_competition_events

meet_id, competition_id = pairs[0]  # first competition
print(f"Fetching events: {meet_id} (id={competition_id})")

events = get_competition_events(competition_id)

# Save full events payload to JSON
out_json = DEBUG_OUT / f"events_{meet_id}_{competition_id}.json"
with out_json.open("w", encoding="utf-8") as f:
    json.dump(events, f, indent=2, default=str)
print(f"Saved events to {out_json}")

# Quick summary
name = events.get("Name") or events.get("OfficialName") or ""
sports = events.get("Sports") or []
n_heats = sum(len(d.get("HeatList") or []) for s in sports for d in (s.get("DisciplineList") or []))
print(f"Competition: {name}")
print(f"Sports: {len(sports)}, total heats: {n_heats}")

Fetching events: oly_2024 (id=2943)
Saved events to /Users/huangrh/workspace/serena/swim-performance/data/raw/debug_build_swim_db/events_oly_2024_2943.json
Competition: Olympic Games Paris 2024
Sports: 5, total heats: 257


In [5]:
# 3. Build disciplines list (SW only) and save to CSV
from src.scraping.world_aquatics_api import iter_disciplines

disciplines = list(iter_disciplines(events))
print(f"Disciplines count (SW only): {len(disciplines)}")

# Save disciplines to CSV
with (DEBUG_OUT / f"disciplines_{meet_id}_{competition_id}.csv").open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["discipline_id", "discipline_name", "gender", "distance"])
    for d in disciplines:
        w.writerow([d.discipline_id, d.discipline_name, d.gender, d.distance])

print(f"First 3 disciplines: {[d.discipline_name for d in disciplines[:3]]}")

Disciplines count (SW only): 35
First 3 disciplines: ["Women's 50m Freestyle", "Women's 100m Freestyle", "Women's 200m Freestyle"]


In [6]:
# 4. Fetch results for first few disciplines (GET /events/{discipline_id})
from src.scraping.world_aquatics_api import get_event_results

LIMIT_DISC = 2  # fetch first 2 disciplines only (for quick debug)
all_results = []

for ref in disciplines[:LIMIT_DISC]:
    rows = get_event_results(ref.discipline_id)
    for r in rows:
        if isinstance(r, dict):
            r = dict(r)
            r["_discipline_name"] = ref.discipline_name
            r["_gender"] = ref.gender
            all_results.append(r)
    print(f"Event {ref.discipline_name}: {len(rows)} results")

print(f"Total raw results: {len(all_results)}")

# Save raw results to CSV (flatten top-level keys)
if all_results:
    keys = set()
    for r in all_results:
        keys.update(k for k in r.keys() if not k.startswith("_"))
    keys = sorted(keys)
    cols = ["_heat_id", "_discipline_name", "_phase_name", "_unit_name"] + keys
    with (DEBUG_OUT / f"results_raw_{meet_id}_{competition_id}.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        w.writerows(all_results)
    print(f"Saved {len(all_results)} raw results to results_raw_*.csv")
else:
    print("WARNING: No results! Check API response structure.")

Event Women's 50m Freestyle: 198 results
Event Women's 100m Freestyle: 98 results
Total raw results: 296
Saved 296 raw results to results_raw_*.csv


In [7]:
# 5. Inspect raw result keys (debug structure)
if all_results:
    r0 = all_results[0]
    print("Keys in first result:")
    for k in sorted(r0.keys()):
        v = r0[k]
        preview = str(v)[:60] + "..." if len(str(v)) > 60 else str(v)
        print(f"  {k}: {preview}")
else:
    print("No results to inspect.")

Keys in first result:
  AthleteResultAge: 30
  BiographyId: 705881
  FirstName: Sarah
  FullName: SJOESTROEM Sarah
  Gender: 1
  GmsId: 100728
  HeatRank: 1
  Lane: 4
  LastName: SJOESTROEM
  MedalTag: G
  NAT: SWE
  PersonId: 9287c8a1-f70b-44b9-9047-65ca5f6c0e90
  Points: 987
  RT: 0.61
  Rank: 1
  ResultId: R1000553
  ScoreboardPhoto: 7a817cc0-92c5-4818-81a6-35107335d271
  Splits: [{'Time': '23.71', 'Distance': '50m', 'Order': 1, 'Different...
  Time: 23.71
  _discipline_name: Women's 50m Freestyle
  _gender: Women
  _heat_id: 6f4ed5ab-502c-4cfb-9400-9bd906b4d80a
  _phase_name: Finals
  _race_date: 2024-08-04
  _race_time: 2024-08-04T16:30:00
  _unit_name: Final


In [8]:
# 6. Normalize results (same logic as build_swim_db)
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.scraping.build_swim_db import _normalize_result_row

normalized = []
for row in all_results:
    if not isinstance(row, dict):
        continue
    ref = next((d for d in disciplines if d.discipline_name == row.get("_discipline_name")), None)
    if not ref:
        continue
    heat_id = row.get("_heat_id") or ""
    unit_name = row.get("_unit_name") or ""
    race_start_time = row.get("_race_time") or row.get("_race_date") or None
    phase = row.get("_phase_name") or ""
    norm = _normalize_result_row(
        row, competition_id, heat_id, ref.discipline_name, ref.gender, phase,
        unit_name=unit_name,
        race_start_time=str(race_start_time) if race_start_time else None,
    )
    normalized.append(norm)

print(f"Normalized {len(normalized)} rows")
if normalized:
    # Save normalized to CSV
    cols = list(normalized[0].keys())
    with (DEBUG_OUT / f"results_normalized_{meet_id}_{competition_id}.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        w.writerows(normalized)
    print(f"Sample: {normalized[0].get('athlete_name')} - {normalized[0].get('time_str')}")

Normalized 296 rows
Sample: Sarah SJOESTROEM - 23.71


In [9]:
# 7. Insert into SQLite (same as build_swim_db)
import sqlite3
from src.scraping.build_swim_db import ensure_schema, _normalize_result_row

db_path = DEBUG_OUT / "swim_db_test.sqlite"
conn = sqlite3.connect(str(db_path))
ensure_schema(conn)

# Insert competition
name = events.get("Name") or events.get("OfficialName") or ""
from_date = (events.get("From") or "")[:10]
to_date = (events.get("To") or "")[:10]
conn.execute(
    "INSERT OR REPLACE INTO competitions (meet_id, competition_id, name, from_date, to_date) VALUES (?,?,?,?,?)",
    (meet_id, competition_id, name, from_date, to_date),
)

# Insert heats (unique heat_ids from results)
seen_heats = set()
for norm in normalized:
    hid = norm["heat_id"]
    key = (competition_id, hid)
    if key not in seen_heats:
        seen_heats.add(key)
        conn.execute(
            "INSERT OR REPLACE INTO heats (competition_id, heat_id, discipline_name, gender, phase, heat_name) VALUES (?,?,?,?,?,?)",
            (competition_id, hid, norm["discipline_name"], norm["gender"], norm["phase"], norm["unit_name"]),
        )

# Insert normalized results
for norm in normalized:
    raw_clean = {}  # skip raw_json for debug
    conn.execute(
        """INSERT INTO results (competition_id, heat_id, discipline_name, gender, phase,
           unit_name, race_start_time, person_id, athlete_name, country_code, time_str, time_ms,
           rank, lane, reaction_time, points, medal_tag, athlete_age, splits_json, raw_json)
           VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
        (
            norm["competition_id"], norm["heat_id"], norm["discipline_name"], norm["gender"], norm["phase"],
            norm["unit_name"], norm["race_start_time"], norm["person_id"] or None, norm["athlete_name"], norm["country_code"],
            norm["time_str"], norm["time_ms"], norm["rank"], norm["lane"],
            norm["reaction_time"], norm["points"], norm["medal_tag"], norm["athlete_age"], norm["splits_json"],
            json.dumps(raw_clean) if raw_clean else None,
        ),
    )

conn.commit()

# Verify
n_comp = conn.execute("SELECT COUNT(*) FROM competitions").fetchone()[0]
n_heat = conn.execute("SELECT COUNT(*) FROM heats").fetchone()[0]
n_res = conn.execute("SELECT COUNT(*) FROM results").fetchone()[0]
conn.close()

print(f"DB: {db_path}")
print(f"  competitions: {n_comp}")
print(f"  heats: {n_heat}")
print(f"  results: {n_res}")

DB: /Users/huangrh/workspace/serena/swim-performance/data/raw/debug_build_swim_db/swim_db_test.sqlite
  competitions: 1
  heats: 7
  results: 296
